In [1]:
# Import necessary libraries
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import csv
import random
import re
import os
import unicodedata
import codecs
import json
import itertools
import math
import time
import numpy as np
from torch.jit import script, trace
from datetime import datetime

# Set random seed for reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Configure device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# Set file paths
corpus_name = "movie-corpus"
corpus_folder = os.path.join("data", corpus_name)

# Create data directory if it doesn't exist
if not os.path.exists("data"):
    os.makedirs("data")
if not os.path.exists(corpus_folder):
    os.makedirs(corpus_folder)

Using cpu device


In [2]:
# Helper function to print lines of a file
def printLines(file, n=10):
    with open(file, 'rb') as datafile:
        lines = datafile.readlines()
    for line in lines[:n]:
        print(line)

# Turn a Unicode string to plain ASCII
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    s = re.sub(r"\s+", r" ", s).strip()
    return s

# Load and process the JSONL file
def loadLinesAndConversations(fileName):
    print(f"Processing {fileName}...")
    lines = {}
    conversations = {}
    
    try:
        with open(fileName, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    lineJson = json.loads(line)
                    # Extract fields for line object
                    lineObj = {}
                    lineObj["lineID"] = lineJson["id"]
                    lineObj["characterID"] = lineJson["speaker"]
                    lineObj["text"] = lineJson["text"]
                    lines[lineObj['lineID']] = lineObj
                    
                    # Extract fields for conversation object
                    if lineJson["conversation_id"] not in conversations:
                        convObj = {}
                        convObj["conversationID"] = lineJson["conversation_id"]
                        convObj["movieID"] = lineJson["meta"]["movie_id"]
                        convObj["lines"] = [lineObj]
                    else:
                        convObj = conversations[lineJson["conversation_id"]]
                        convObj["lines"].append(lineObj)
                    conversations[convObj["conversationID"]] = convObj
                except json.JSONDecodeError:
                    print(f"Warning: Couldn't parse line as JSON: {line[:50]}...")
                    continue
                except KeyError as e:
                    print(f"Warning: Key error {e} in line: {line[:50]}...")
                    continue
    except FileNotFoundError:
        print(f"File {fileName} not found. Please download the dataset first.")
        return {}, {}
    
    return lines, conversations

# Create Q&A pairs from conversations
def extractSentencePairs(conversations):
    qa_pairs = []
    for conversation in conversations.values():
        # Sort lines by ID to ensure correct order
        sorted_lines = sorted(conversation["lines"], key=lambda x: x["lineID"])
        
        # Iterate over all the lines of the conversation
        for i in range(len(sorted_lines) - 1):  # We ignore the last line (no answer for it)
            inputLine = sorted_lines[i]["text"].strip()
            targetLine = sorted_lines[i+1]["text"].strip()
            # Filter wrong samples (if one of the lists is empty)
            if inputLine and targetLine:
                qa_pairs.append([inputLine, targetLine])
    return qa_pairs

In [3]:
# Prepare the data
def prepareData(corpus_folder, utterances_file="utterances.jsonl"):
    # Define path to JSONL file
    datafile = os.path.join(corpus_folder, utterances_file)
    
    # Define path to new formatted file
    formatted_file = os.path.join(corpus_folder, "formatted_movie_lines.txt")
    
    # Load lines and conversations
    print("\nProcessing corpus into lines and conversations...")
    lines, conversations = loadLinesAndConversations(datafile)
    
    # Extract pairs
    print("\nExtracting pairs...")
    pairs = extractSentencePairs(conversations)
    
    # Write new csv file
    print("\nWriting newly formatted file...")
    delimiter = '\t'
    # Unescape the delimiter
    delimiter = str(codecs.decode(delimiter, "unicode_escape"))
    
    with open(formatted_file, 'w', encoding='utf-8') as outputfile:
        writer = csv.writer(outputfile, delimiter=delimiter, lineterminator='\n')
        for pair in pairs:
            writer.writerow(pair)
    
    # Print a sample of lines
    print("\nSample lines from file:")
    printLines(formatted_file)
    
    return formatted_file

In [4]:
# Default word tokens
PAD_token = 0  # Used for padding short sentences
SOS_token = 1  # Start-of-sentence token
EOS_token = 2  # End-of-sentence token
UNK_token = 3  # Unknown word token

class Vocabulary:
    def __init__(self, name):
        self.name = name
        self.trimmed = False
        self.word2index = {"PAD": PAD_token, "SOS": SOS_token, "EOS": EOS_token, "UNK": UNK_token}
        self.word2count = {}
        self.index2word = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS", UNK_token: "UNK"}
        self.num_words = 4  # Count default tokens
        
    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)
            
    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.num_words
            self.word2count[word] = 1
            self.index2word[self.num_words] = word
            self.num_words += 1
        else:
            self.word2count[word] += 1
            
    def trim(self, min_count):
        if self.trimmed:
            return
        self.trimmed = True
        
        keep_words = []
        for k, v in self.word2count.items():
            if v >= min_count:
                keep_words.append(k)
                
        print('keep_words {} / {} = {:.4f}'.format(
            len(keep_words), len(self.word2index), len(keep_words) / len(self.word2index)
        ))
        
        # Reinitialize dictionaries
        self.word2index = {"PAD": PAD_token, "SOS": SOS_token, "EOS": EOS_token, "UNK": UNK_token}
        self.word2count = {}
        self.index2word = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS", UNK_token: "UNK"}
        self.num_words = 4  # Count default tokens
        
        for word in keep_words:
            self.addWord(word)

In [5]:
# Maximum sentence length to consider
MAX_LENGTH = 20  # Increased to allow for more natural dialogues

# Read query/response pairs and return a voc object
def readVocs(datafile, corpus_name):
    print("Reading lines...")
    # Read the file and split into lines
    lines = open(datafile, encoding='utf-8').read().strip().split('\n')
    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    vocab = Vocabulary(corpus_name)
    return vocab, pairs

# Returns True if both sentences in a pair 'p' are under the MAX_LENGTH threshold
def filterPair(p):
    # Input sequences need to preserve the last word for EOS token
    return len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) < MAX_LENGTH

# Filter pairs using the filterPair condition
def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

# Using the functions defined above, return a populated voc object and pairs list
def loadPrepareData(corpus_folder, corpus_name, datafile):
    print("Start preparing training data...")
    vocab, pairs = readVocs(datafile, corpus_name)
    print("Read {!s} sentence pairs".format(len(pairs)))
    pairs = filterPairs(pairs)
    print("Trimmed to {!s} sentence pairs".format(len(pairs)))
    print("Counting words...")
    for pair in pairs:
        vocab.addSentence(pair[0])
        vocab.addSentence(pair[1])
    print("Counted words:", vocab.num_words)
    return vocab, pairs

# Minimum word count threshold for trimming
MIN_COUNT = 3

def trimRareWords(vocab, pairs, min_count):
    # Trim words used under the MIN_COUNT from the vocab
    vocab.trim(min_count)
    # Filter out pairs with trimmed words
    keep_pairs = []
    for pair in pairs:
        input_sentence = pair[0]
        output_sentence = pair[1]
        keep_input = True
        keep_output = True
        
        # Check input sentence
        for word in input_sentence.split(' '):
            if word not in vocab.word2index:
                keep_input = False
                break
                
        # Check output sentence
        for word in output_sentence.split(' '):
            if word not in vocab.word2index:
                keep_output = False
                break
                
        # Only keep pairs that do not contain trimmed words
        if keep_input and keep_output:
            keep_pairs.append(pair)
            
    print("Trimmed from {} pairs to {}, {:.4f} of total".format(
        len(pairs), len(keep_pairs), len(keep_pairs) / len(pairs)))
    
    return keep_pairs

# Prepare data for model
def prepareDataForModel(corpus_folder, corpus_name):
    # Get datafile path
    datafile = os.path.join(corpus_folder, "formatted_movie_lines.txt")
    
    # Load and prepare data
    vocab, pairs = loadPrepareData(corpus_folder, corpus_name, datafile)
    
    # Trim rare words
    pairs = trimRareWords(vocab, pairs, MIN_COUNT)
    
    return vocab, pairs

In [6]:
# Helper functions for processing data for the model
def indexesFromSentence(vocab, sentence):
    # For unknown words, use UNK_token
    return [vocab.word2index.get(word, UNK_token) for word in sentence.split(' ')] + [EOS_token]

def zeroPadding(l, fillvalue=PAD_token):
    return list(itertools.zip_longest(*l, fillvalue=fillvalue))

def binaryMatrix(l, value=PAD_token):
    m = []
    for i, seq in enumerate(l):
        m.append([])
        for token in seq:
            if token == PAD_token:
                m[i].append(0)
            else:
                m[i].append(1)
    return m

# Returns padded input sequence tensor and lengths
def inputVar(l, vocab):
    indexes_batch = [indexesFromSentence(vocab, sentence) for sentence in l]
    lengths = torch.tensor([len(indexes) for indexes in indexes_batch])
    padList = zeroPadding(indexes_batch)
    padVar = torch.LongTensor(padList)
    return padVar, lengths

# Returns padded target sequence tensor, padding mask, and max target length
def outputVar(l, vocab):
    indexes_batch = [indexesFromSentence(vocab, sentence) for sentence in l]
    max_target_len = max([len(indexes) for indexes in indexes_batch])
    padList = zeroPadding(indexes_batch)
    mask = binaryMatrix(padList)
    mask = torch.BoolTensor(mask)
    padVar = torch.LongTensor(padList)
    return padVar, mask, max_target_len

# Returns all items for a given batch of pairs
def batch2TrainData(vocab, pair_batch):
    # Sort by length (descending)
    pair_batch.sort(key=lambda x: len(x[0].split(" ")), reverse=True)
    input_batch, output_batch = [], []
    for pair in pair_batch:
        input_batch.append(pair[0])
        output_batch.append(pair[1])
    inp, lengths = inputVar(input_batch, vocab)
    output, mask, max_target_len = outputVar(output_batch, vocab)
    # Return tensors on the correct device
    inp = inp.to(device)
    lengths = lengths.to(device)
    output = output.to(device)
    mask = mask.to(device)
    return inp, lengths, output, mask, max_target_len

In [7]:
# Define the Encoder network
class EncoderRNN(nn.Module):
    def __init__(self, hidden_size, embedding, n_layers=1, dropout=0):
        super(EncoderRNN, self).__init__()
        self.n_layers = n_layers
        self.hidden_size = hidden_size
        self.embedding = embedding
        
        # GRU layer - using bidirectional for better context capture
        self.gru = nn.GRU(
            hidden_size, 
            hidden_size, 
            n_layers,
            dropout=(0 if n_layers == 1 else dropout), 
            bidirectional=True
        )
        
        # Linear layer to combine bidirectional outputs
        self.fc = nn.Linear(hidden_size * 2, hidden_size)
        
    def forward(self, input_seq, input_lengths, hidden=None):
        # Convert word indexes to embeddings
        embedded = self.embedding(input_seq)
        
        # Pack padded batch of sequences for RNN module
        packed = nn.utils.rnn.pack_padded_sequence(embedded, input_lengths.cpu())
        
        # Forward pass through GRU
        outputs, hidden = self.gru(packed, hidden)
        
        # Unpack padding
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs)
        
        # Sum bidirectional GRU outputs
        outputs = outputs[:, :, :self.hidden_size] + outputs[:, :, self.hidden_size:]
        
        return outputs, hidden

In [8]:
# Define attention mechanisms
class Attn(nn.Module):
    def __init__(self, method, hidden_size):
        super(Attn, self).__init__()
        self.method = method
        self.hidden_size = hidden_size
        
        if self.method == 'general':
            self.attn = nn.Linear(self.hidden_size, hidden_size)
        elif self.method == 'concat':
            self.attn = nn.Linear(self.hidden_size * 2, hidden_size)
            self.v = nn.Parameter(torch.FloatTensor(hidden_size))
        elif self.method == 'scaled_dot':
            # Scaled dot-product attention from "Attention Is All You Need"
            self.sqrt_dim = torch.sqrt(torch.FloatTensor([hidden_size])).to(device)
            
    def dot_score(self, hidden, encoder_output):
        return torch.sum(hidden * encoder_output, dim=2)
        
    def general_score(self, hidden, encoder_output):
        energy = self.attn(encoder_output)
        return torch.sum(hidden * energy, dim=2)
        
    def concat_score(self, hidden, encoder_output):
        energy = self.attn(torch.cat((hidden.expand(encoder_output.size(0), -1, -1), 
                                     encoder_output), 2)).tanh()
        return torch.sum(self.v * energy, dim=2)
    
    def scaled_dot_score(self, hidden, encoder_output):
        # Scaled dot-product attention
        return torch.sum(hidden * encoder_output, dim=2) / self.sqrt_dim
        
    def forward(self, hidden, encoder_outputs):
        # Calculate the attention weights (energies) based on the given method
        if self.method == 'general':
            attn_energies = self.general_score(hidden, encoder_outputs)
        elif self.method == 'concat':
            attn_energies = self.concat_score(hidden, encoder_outputs)
        elif self.method == 'dot':
            attn_energies = self.dot_score(hidden, encoder_outputs)
        elif self.method == 'scaled_dot':
            attn_energies = self.scaled_dot_score(hidden, encoder_outputs)
            
        # Transpose max_length and batch_size dimensions
        attn_energies = attn_energies.t()
        
        # Return the softmax normalized probability scores (with added dimension)
        return F.softmax(attn_energies, dim=1).unsqueeze(1)

In [9]:
# Define the Decoder network with attention
class LuongAttnDecoderRNN(nn.Module):
    def __init__(self, attn_model, embedding, hidden_size, output_size, n_layers=1, dropout=0.1):
        super(LuongAttnDecoderRNN, self).__init__()
        
        # Keep for reference
        self.attn_model = attn_model
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout
        
        # Define layers
        self.embedding = embedding
        self.embedding_dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden_size, hidden_size, n_layers, dropout=(0 if n_layers == 1 else dropout))
        self.concat = nn.Linear(hidden_size * 2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        
        # Choose attention model
        self.attn = Attn(attn_model, hidden_size)
        
        # Add layer normalization for better stability
        self.layer_norm = nn.LayerNorm(hidden_size)
        
    def forward(self, input_step, last_hidden, encoder_outputs):
        # Note: we run this one step (word) at a time
        # Get embedding of current input word
        embedded = self.embedding(input_step)
        embedded = self.embedding_dropout(embedded)
        
        # Forward through unidirectional GRU
        rnn_output, hidden = self.gru(embedded, last_hidden)
        
        # Calculate attention weights from the current GRU output
        attn_weights = self.attn(rnn_output, encoder_outputs)
        
        # Multiply attention weights to encoder outputs to get new "weighted sum" context vector
        context = attn_weights.bmm(encoder_outputs.transpose(0, 1))
        
        # Concatenate weighted context vector and GRU output using Luong eq. 5
        rnn_output = rnn_output.squeeze(0)
        context = context.squeeze(1)
        concat_input = torch.cat((rnn_output, context), 1)
        concat_output = torch.tanh(self.concat(concat_input))
        
        # Apply layer normalization
        concat_output = self.layer_norm(concat_output)
        
        # Predict next word using Luong eq. 6
        output = self.out(concat_output)
        output = F.softmax(output, dim=1)
        
        # Return output and final hidden state
        return output, hidden

In [10]:
# Define loss function
def maskNLLLoss(inp, target, mask):
    nTotal = mask.sum()
    crossEntropy = -torch.log(torch.gather(inp, 1, target.view(-1, 1)).squeeze(1) + 1e-10)  # Added epsilon for stability
    loss = crossEntropy.masked_select(mask).mean()
    loss = loss.to(device)
    return loss, nTotal.item()

In [11]:
# Define training function with scheduled sampling and learning rate decay
def train(input_variable, lengths, target_variable, mask, max_target_len, encoder, decoder, embedding,
          encoder_optimizer, decoder_optimizer, batch_size, clip, epoch, n_epochs, teacher_forcing_ratio=0.8, max_length=MAX_LENGTH):
    
    # Zero gradients
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    
    # Initialize variables
    loss = 0
    print_losses = []
    n_totals = 0
    
    # Forward pass through encoder
    encoder_outputs, encoder_hidden = encoder(input_variable, lengths)
    
    # Create initial decoder input (start with SOS tokens for each sentence)
    decoder_input = torch.LongTensor([[SOS_token for _ in range(batch_size)]])
    decoder_input = decoder_input.to(device)
    
    # Set initial decoder hidden state to the encoder's final hidden state
    decoder_hidden = encoder_hidden[:decoder.n_layers]
    
    # Implement scheduled sampling - decrease teacher forcing ratio over time
    adjusted_teaching_ratio = max(0.2, teacher_forcing_ratio - 0.1 * (epoch / n_epochs))
    use_teacher_forcing = True if random.random() < adjusted_teaching_ratio else False
    
    # Forward batch of sequences through decoder one time step at a time
    if use_teacher_forcing:
        for t in range(max_target_len):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden, encoder_outputs
            )
            # Teacher forcing: next input is current target
            decoder_input = target_variable[t].view(1, -1)
            # Calculate and accumulate loss
            mask_loss, nTotal = maskNLLLoss(decoder_output, target_variable[t], mask[t])
            loss += mask_loss
            print_losses.append(mask_loss.item() * nTotal)
            n_totals += nTotal
    else:
        for t in range(max_target_len):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden, encoder_outputs
            )
            # No teacher forcing: next input is decoder's own current output
            _, topi = decoder_output.topk(1)
            decoder_input = torch.LongTensor([[topi[i][0] for i in range(batch_size)]])
            decoder_input = decoder_input.to(device)
            # Calculate and accumulate loss
            mask_loss, nTotal = maskNLLLoss(decoder_output, target_variable[t], mask[t])
            loss += mask_loss
            print_losses.append(mask_loss.item() * nTotal)
            n_totals += nTotal
            
    # Perform backpropagation
    loss.backward()
    
    # Clip gradients: gradients are modified in place
    _ = nn.utils.clip_grad_norm_(encoder.parameters(), clip)
    _ = nn.utils.clip_grad_norm_(decoder.parameters(), clip)
    
    # Adjust model weights
    encoder_optimizer.step()
    decoder_optimizer.step()
    
    return sum(print_losses) / n_totals

In [12]:
# Training iterations function with learning rate scheduling
def trainIters(model_name, vocab, pairs, encoder, decoder, encoder_optimizer, decoder_optimizer, embedding,
              encoder_n_layers, decoder_n_layers, save_dir, n_iteration, batch_size, print_every,
              save_every, clip, corpus_name, loadFilename=None, teacher_forcing_ratio=0.8, hidden_size=768,
              learning_rate=0.0001, n_epochs=20):
    
    # Load batches for each iteration
    n_batches = n_iteration // batch_size
    training_batches = [batch2TrainData(vocab, [random.choice(pairs) for _ in range(batch_size)])
                      for _ in range(n_batches)]
    
    # Initializations
    print('Initializing...')
    start_iteration = 1
    print_loss = 0
    
    # Load checkpoint if requested
    if loadFilename:
        checkpoint = torch.load(loadFilename)
        start_iteration = checkpoint['iteration'] + 1
    
    # Learning rate scheduler
    encoder_scheduler = torch.optim.lr_scheduler.StepLR(encoder_optimizer, step_size=1, gamma=0.95)
    decoder_scheduler = torch.optim.lr_scheduler.StepLR(decoder_optimizer, step_size=1, gamma=0.95)
    
    # Create directory for saving model
    directory = os.path.join(save_dir, model_name, corpus_name, '{}-{}_{}'.format(
        encoder_n_layers, decoder_n_layers, hidden_size))
    if not os.path.exists(directory):
        os.makedirs(directory)
    
    # Training loop
    print("Training...")
    start_time = time.time()
    
    for epoch in range(n_epochs):
        print(f"\nEpoch {epoch+1}/{n_epochs}")
        
        # Get learning rates
        encoder_lr = encoder_optimizer.param_groups[0]['lr']
        decoder_lr = decoder_optimizer.param_groups[0]['lr']
        print(f"Learning rates: encoder={encoder_lr:.6f}, decoder={decoder_lr:.6f}")
        
        for i, training_batch in enumerate(training_batches):
            # Progress info
            iteration = epoch * len(training_batches) + i + 1
            
            # Extract fields from batch
            input_variable, lengths, target_variable, mask, max_target_len = training_batch
            
            # Run a training iteration with batch
            loss = train(input_variable, lengths, target_variable, mask, max_target_len, encoder,
                       decoder, embedding, encoder_optimizer, decoder_optimizer, batch_size, clip, 
                       epoch, n_epochs, teacher_forcing_ratio)
            print_loss += loss
            
            # Print progress
            if iteration % print_every == 0:
                print_loss_avg = print_loss / print_every
                elapsed = time.time() - start_time
                print("Iteration: {}; Percent complete: {:.1f}%; Average loss: {:.4f}; Time elapsed: {:.1f}m".format(
                    iteration, iteration / (n_epochs * len(training_batches)) * 100, print_loss_avg, elapsed / 60))
                print_loss = 0
                
            # Save checkpoint
            if (iteration % save_every == 0):
                torch.save({
                    'iteration': iteration,
                    'en': encoder.state_dict(),
                    'de': decoder.state_dict(),
                    'en_opt': encoder_optimizer.state_dict(),
                    'de_opt': decoder_optimizer.state_dict(),
                    'loss': loss,
                    'voc_dict': vocab.__dict__,
                    'embedding': embedding.state_dict(),
                    'epoch': epoch
                }, os.path.join(directory, '{}_{}.tar'.format(iteration, 'checkpoint')))
        
        # Step the schedulers
        encoder_scheduler.step()
        decoder_scheduler.step()
    
    # Final save
    torch.save({
        'iteration': iteration,
        'en': encoder.state_dict(),
        'de': decoder.state_dict(),
        'en_opt': encoder_optimizer.state_dict(),
        'de_opt': decoder_optimizer.state_dict(),
        'loss': loss,
        'voc_dict': vocab.__dict__,
        'embedding': embedding.state_dict(),
        'epoch': n_epochs
    }, os.path.join(directory, 'final_model.tar'))
    
    print(f"\nTraining completed in {(time.time() - start_time) / 60:.1f} minutes")
    return encoder, decoder

In [13]:
# Define beam search decoder for better responses
class BeamSearchDecoder(nn.Module):
    def __init__(self, encoder, decoder, beam_size=5):
        super(BeamSearchDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beam_size = beam_size
        
    def forward(self, input_seq, input_length, max_length):
        # Forward input through encoder model
        encoder_outputs, encoder_hidden = self.encoder(input_seq, input_length)
        
        # Prepare encoder's final hidden layer to be first hidden input to the decoder
        decoder_hidden = encoder_hidden[:self.decoder.n_layers]
        
        # Initialize decoder input with SOS_token
        decoder_input = torch.ones(1, 1, device=device, dtype=torch.long) * SOS_token
        
        # Initialize beam search
        # Each beam contains (sequence, score, hidden)
        beams = [(torch.zeros([0], device=device, dtype=torch.long), 0.0, decoder_hidden)]
        
        # Iteratively decode one word token at a time
        for _ in range(max_length):
            new_beams = []
            
            # Expand each current beam
            for sequence, score, hidden in beams:
                # Stop expanding if the last token was EOS
                if sequence.size(0) > 0 and sequence[-1].item() == EOS_token:
                    new_beams.append((sequence, score, hidden))
                    continue
                
                # Get the last token
                if sequence.size(0) > 0:
                    decoder_input = sequence[-1].view(1, -1)
                else:
                    decoder_input = torch.ones(1, 1, device=device, dtype=torch.long) * SOS_token
                
                # Forward pass through decoder
                decoder_output, new_hidden = self.decoder(decoder_input, hidden, encoder_outputs)
                
                # Get top-k tokens
                topk_scores, topk_tokens = torch.topk(decoder_output, self.beam_size)
                
                # Create new beams
                for i in range(self.beam_size):
                    token = topk_tokens[0][i].view(-1)
                    log_prob = torch.log(topk_scores[0][i]).item()
                    
                    new_sequence = torch.cat((sequence, token), dim=0)
                    new_score = score + log_prob
                    
                    new_beams.append((new_sequence, new_score, new_hidden))
            
            # Sort and keep only top-k beams
            beams = sorted(new_beams, key=lambda x: x[1] / (len(x[0]) + 1), reverse=True)[:self.beam_size]
            
            # Check if all beams end with EOS
            if all(beam[0].size(0) > 0 and beam[0][-1].item() == EOS_token for beam in beams):
                break
                
        # Return the best beam (sequence, score)
        return beams[0][0], torch.tensor([beams[0][1]])

In [14]:
# Function to evaluate text input with beam search
def evaluate(encoder, decoder, searcher, vocab, sentence, max_length=MAX_LENGTH):
    # Format input sentence as a batch
    # words -> indexes
    indexes_batch = [indexesFromSentence(vocab, sentence)]
    
    # Create lengths tensor
    lengths = torch.tensor([len(indexes) for indexes in indexes_batch])
    
    # Transpose dimensions of batch to match models' expectations
    input_batch = torch.LongTensor(indexes_batch).transpose(0, 1)
    
    # Use appropriate device
    input_batch = input_batch.to(device)
    lengths = lengths.to(device)
    
    # Decode sentence with searcher
    tokens, scores = searcher(input_batch, lengths, max_length)
    
    # indexes -> words
    decoded_words = [vocab.index2word[token.item()] for token in tokens]
    
    # Remove EOS and PAD tokens
    if EOS_token in [token.item() for token in tokens]:
        eos_idx = [token.item() for token in tokens].index(EOS_token)
        decoded_words = decoded_words[:eos_idx]
    
    return decoded_words

In [15]:
# Import for sentiment analysis
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

# Try to download NLTK resources, handle offline case
try:
    nltk.download('vader_lexicon', quiet=True)
    sentiment_analyzer = SentimentIntensityAnalyzer()
except:
    print("NLTK resources not available. Sentiment analysis will be limited.")
    sentiment_analyzer = None

# A more user-friendly chat interface with history and context
def enhanced_chat_interface(encoder, decoder, searcher, vocab):
    # Initialize chat history and context tracking
    history = []
    context_window = 5  # Number of recent exchanges to consider for context
    
    print("\n=== Enhanced Movie Dialogue Chatbot ===")
    print("Type 'quit' to exit, 'clear' to clear chat history, or 'help' for more commands.")
    print("Bot: Hello! I'm your movie dialogue assistant. What would you like to talk about?")
    
    # Add greeting to history
    history.append({"role": "bot", "message": "Hello! I'm your movie dialogue assistant. What would you like to talk about?"})
    
    while True:
        try:
            # Get input
            user_input = input("You: ")
            
            # Check for commands
            if user_input.lower() in ['quit', 'exit', 'q']:
                print("Bot: Goodbye! Have a great day!")
                break
                
            if user_input.lower() == 'clear':
                history = []
                print("Bot: Chat history cleared.")
                continue
                
            if user_input.lower() == 'help':
                print("\nAvailable commands:")
                print("'quit' or 'exit' - End the conversation")
                print("'clear' - Clear chat history")
                print("'history' - Show recent conversation history")
                print("'save' - Save the conversation to a file")
                continue
                
            if user_input.lower() == 'history':
                print("\nRecent conversation:")
                for i, item in enumerate(history[-10:]):  # Show last 10 exchanges
                    print(f"{item['role'].capitalize()}: {item['message']}")
                continue
                
            if user_input.lower() == 'save':
                filename = f"chat_history_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
                with open(filename, 'w', encoding='utf-8') as f:
                    for item in history:
                        f.write(f"{item['role'].capitalize()}: {item['message']}\n")
                print(f"Bot: Conversation saved to {filename}")
                continue
                
            # Add to history
            history.append({"role": "user", "message": user_input})
            
            # Analyze sentiment if available
            sentiment_score = 0
            if sentiment_analyzer:
                sentiment_score = sentiment_analyzer.polarity_scores(user_input)['compound']
            
            # Create context from recent history
            context = ""
            if len(history) > 1:
                recent_history = history[-min(context_window*2+1, len(history)):-1]
                for item in recent_history:
                    if item['role'] == 'user':
                        context += item['message'] + " "
            
            # Normalize sentence
            normalized_input = normalizeString(user_input)
            
            # Evaluate with model
            output_words = evaluate(encoder, decoder, searcher, vocab, normalized_input)
            output_words = [x for x in output_words if not (x == 'EOS' or x == 'PAD')]
            
            # Handle empty or very short responses with contextual fallbacks
            if len(output_words) < 2:
                # Choose a fallback based on sentiment and context
                if sentiment_score > 0.3:  # Positive sentiment
                    fallbacks = [
                        "That sounds great! Tell me more about it.",
                        "I'm glad to hear that! What else is on your mind?",
                        "Wonderful! I'd love to hear more details.",
                        "That's excellent! Let's explore that further.",
                        "Fantastic! Can you elaborate on that?"
                    ]
                elif sentiment_score < -0.3:  # Negative sentiment
                    fallbacks = [
                        "I understand that can be challenging. Would you like to talk more about it?",
                        "I'm sorry to hear that. How can I help?",
                        "That sounds difficult. Let's discuss it further if you'd like.",
                        "I appreciate you sharing that with me. Would you like to continue on this topic?",
                        "Thank you for being open about that. What else is on your mind?"
                    ]
                else:  # Neutral sentiment
                    fallbacks = [
                        "I see. Could you tell me more about that?",
                        "Interesting! What are your thoughts on this?",
                        "I'd like to understand better. Can you elaborate?",
                        "Let's explore that idea. What aspects interest you most?",
                        "That's an interesting point. What else would you like to discuss?"
                    ]
                bot_response = random.choice(fallbacks)
            else:
                bot_response = ' '.join(output_words)
                
            # Apply some post-processing to improve response quality
            bot_response = bot_response.capitalize()
            if not any(bot_response.endswith(p) for p in ['.', '!', '?']):
                bot_response += '.'
                
            # Print response
            print(f"Bot: {bot_response}")
            
            # Add to history
            history.append({"role": "bot", "message": bot_response})
            
        except KeyError:
            bot_response = "I don't know some of those words. Could you rephrase that?"
            print(f"Bot: {bot_response}")
            history.append({"role": "bot", "message": bot_response})
            
        except Exception as e:
            print(f"Error: {str(e)}")
            bot_response = "Something went wrong. Let's try again."
            print(f"Bot: {bot_response}")
            history.append({"role": "bot", "message": bot_response})

In [16]:
# Main function to run the enhanced chatbot
def run_enhanced_chatbot(load_checkpoint=None):
    # Model configuration
    model_name = 'enhanced_movie_chatbot'
    attn_model = 'general'  # Using general attention for better performance
    hidden_size = 768       # Increased hidden size for better representations
    encoder_n_layers = 3    # Deeper network for more complex patterns
    decoder_n_layers = 3
    dropout = 0.2           # Increased dropout for better generalization
    batch_size = 64
    
    # Set paths
    save_dir = os.path.join("data", "save")
    corpus_folder = os.path.join("data", corpus_name)
    
    # Ensure directories exist
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    if not os.path.exists(corpus_folder):
        os.makedirs(corpus_folder)
    
    # Process data if needed
    print("\nChecking data...")
    formatted_file = os.path.join(corpus_folder, "formatted_movie_lines.txt")
    
    if not os.path.exists(formatted_file):
        # Determine path to original corpus
        utterances_file = "utterances.jsonl"
        user_path = input("Please enter the path to your utterances.jsonl file: ")
        
        if os.path.exists(user_path):
            # Copy the file to our corpus folder
            import shutil
            target_path = os.path.join(corpus_folder, utterances_file)
            shutil.copy2(user_path, target_path)
            
            # Process the data
            prepareData(corpus_folder, utterances_file)
        else:
            print(f"Error: {user_path} not found.")
            print("Please download the Cornell Movie-Dialogs Corpus and try again.")
            return
    else:
        print(f"Using existing {formatted_file}")
    
    # Load and prepare data for model
    vocab, pairs = prepareDataForModel(corpus_folder, corpus_name)
    
    print("\nSample dialogue pairs:")
    for i in range(5):
        print(random.choice(pairs))
    
    # Initialize model
    print('\nInitializing model...')
    
    # Initialize word embeddings
    embedding = nn.Embedding(vocab.num_words, hidden_size)
    
    # Initialize encoder & decoder models
    encoder = EncoderRNN(hidden_size, embedding, encoder_n_layers, dropout)
    decoder = LuongAttnDecoderRNN(attn_model, embedding, hidden_size, vocab.num_words, decoder_n_layers, dropout)
    
    # Load checkpoint if provided
    if load_checkpoint:
        try:
            print(f"Loading checkpoint {load_checkpoint}...")
            checkpoint = torch.load(load_checkpoint)
            encoder.load_state_dict(checkpoint['en'])
            decoder.load_state_dict(checkpoint['de'])
            embedding.load_state_dict(checkpoint['embedding'])
            vocab.__dict__ = checkpoint['voc_dict']
            print("Checkpoint loaded successfully!")
        except Exception as e:
            print(f"Error loading checkpoint: {str(e)}")
            return
    else:
        # Configure training
        clip = 50.0
        teacher_forcing_ratio = 0.7
        learning_rate = 0.0001
        decoder_learning_ratio = 5.0
        n_iteration = 4000
        print_every = 100
        save_every = 500
        n_epochs = 50  # Increased for better convergence
        
        # Set dropout layers to train mode
        encoder.train()
        decoder.train()
        
        # Initialize optimizers
        print('Initializing optimizers...')
        encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
        decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate * decoder_learning_ratio)
        
        # Move models to device
        encoder = encoder.to(device)
        decoder = decoder.to(device)
        
        # Train model
        print("\nStarting training...")
        encoder, decoder = trainIters(
            model_name, vocab, pairs, encoder, decoder, 
            encoder_optimizer, decoder_optimizer,
            embedding, encoder_n_layers, decoder_n_layers, 
            save_dir, n_iteration, batch_size,
            print_every, save_every, clip, corpus_name, 
            load_checkpoint, teacher_forcing_ratio, hidden_size,
            learning_rate, n_epochs
        )
    
    # Set dropout layers to eval mode
    encoder.eval()
    decoder.eval()
    
    # Initialize search module
    print("\nInitializing beam search decoder...")
    searcher = BeamSearchDecoder(encoder, decoder, beam_size=5)
    
    # Begin chatting
    print("\nStarting chat interface...")
    enhanced_chat_interface(encoder, decoder, searcher, vocab)
    
    return encoder, decoder, searcher, vocab

In [17]:
# Function for evaluating the chatbot's performance
def evaluate_chatbot_performance(encoder, decoder, searcher, vocab, test_pairs, beam_size=5):
    print("\nEvaluating chatbot performance...")
    
    # Set models to evaluation mode
    encoder.eval()
    decoder.eval()
    
    # Metrics
    total_samples = min(len(test_pairs), 100)  # Evaluate on up to 100 samples
    bleu_scores = []
    response_lengths = []
    diversity_scores = []
    
    try:
        from nltk.translate.bleu_score import sentence_bleu
        from nltk.tokenize import word_tokenize
        nltk.download('punkt', quiet=True)
    except:
        print("NLTK not available. Skipping BLEU score calculation.")
        sentence_bleu = None
    
    # Sample random test pairs
    test_samples = random.sample(test_pairs, total_samples)
    
    for i, pair in enumerate(test_samples):
        # Input sentence from the test pair
        input_sentence = pair[0]
        target_sentence = pair[1]
        
        # Generate response
        output_words = evaluate(encoder, decoder, searcher, vocab, input_sentence)
        output_words = [x for x in output_words if not (x == 'EOS' or x == 'PAD')]
        
        # Measure response length
        response_length = len(output_words)
        response_lengths.append(response_length)
        
        # Calculate BLEU score if available
        if sentence_bleu and response_length > 0:
            try:
                reference = [word_tokenize(target_sentence.lower())]
                candidate = word_tokenize(' '.join(output_words).lower())
                
                if len(candidate) > 0:
                    bleu_score = sentence_bleu(reference, candidate)
                    bleu_scores.append(bleu_score)
            except:
                pass
        
        # Calculate diversity (unique words / total words)
        if response_length > 0:
            unique_words = len(set(output_words))
            diversity = unique_words / response_length
            diversity_scores.append(diversity)
        
        # Print progress
        if (i + 1) % 10 == 0:
            print(f"Evaluated {i + 1}/{total_samples} samples")
    
    # Calculate and print final metrics
    print("\nChatbot Performance Metrics:")
    
    if bleu_scores:
        avg_bleu = sum(bleu_scores) / len(bleu_scores)
        print(f"Average BLEU score: {avg_bleu:.4f}")
    
    if response_lengths:
        avg_length = sum(response_lengths) / len(response_lengths)
        print(f"Average response length: {avg_length:.2f} words")
    
    if diversity_scores:
        avg_diversity = sum(diversity_scores) / len(diversity_scores)
        print(f"Average lexical diversity: {avg_diversity:.4f}")
    
    print("\nEvaluation complete!")

# Main execution - uncomment the line below to run the chatbot
# run_enhanced_chatbot()

# To evaluate the model after training, uncomment the following:
# evaluate_chatbot_performance(encoder, decoder, searcher, vocab, test_pairs)

if __name__ == "__main__":
    print("Enhanced Movie Dialogue Chatbot")
    print("1. Train and run new chatbot")
    print("2. Load existing model")
    
    choice = input("Enter your choice (1/2): ")
    
    if choice == "1":
        run_enhanced_chatbot()
    elif choice == "2":
        checkpoint_path = input("Enter path to checkpoint file: ")
        if os.path.exists(checkpoint_path):
            run_enhanced_chatbot(checkpoint_path)
        else:
            print(f"Error: {checkpoint_path} not found.")
    else:
        print("Invalid choice. Exiting.")

Enhanced Movie Dialogue Chatbot
1. Train and run new chatbot
2. Load existing model

Checking data...

Processing corpus into lines and conversations...
Processing data\movie-corpus\utterances.jsonl...

Extracting pairs...

Writing newly formatted file...

Sample lines from file:
b'They do to!\tThey do not!\r\n'
b'She okay?\tI hope so.\r\n'
b"Wow\tLet's go.\r\n"
b'"I\'m kidding.  You know how sometimes you just become this ""persona""?  And you don\'t know how to quit?"\tNo\r\n'
b"No\tOkay -- you're gonna need to learn how to lie.\r\n"
b"I figured you'd get to the good stuff eventually.\tWhat good stuff?\r\n"
b'What good stuff?\t"The ""real you""."\r\n'
b'"The ""real you""."\tLike my fear of wearing pastels?\r\n'
b'do you listen to this crap?\tWhat crap?\r\n'
b"What crap?\tMe.  This endless ...blonde babble. I'm like, boring myself.\r\n"
Start preparing training data...
Reading lines...
Read 221282 sentence pairs
Trimmed to 145492 sentence pairs
Counting words...
Counted words: 33171
k